# Phase 4 (comparison): the leakage audit (optional)

The same pipeline under the common intra-patient protocol: DS1 and DS2 beats pooled and split at random
per class (same counts per part as the inter-patient protocol). The generators frozen at the phase 3 gate
are retrained on the intra-patient training beats; the cnn is trained with none, cVAE and WGAN-GP at
medium strength on the grid seeds and tested on the intra-patient test beats. Never used for any decision.

It reads DS2, so it runs **only after the final evaluation**: the branch you run must hold the evaluation
entry in `docs/TEST_ACCESS_LOG.md` and `results/final_eval/summary.json`. Otherwise it refuses (exit code 2).

Before running (details in `kaggle/README.md`):

1. Settings: **Internet on**. Accelerator: **GPU**. For long runs use Save Version > Save & Run All.
2. Add-ons > Secrets: `GITHUB_TOKEN` (the repo is private), ticked for this notebook.
3. Set `CONFIRM_AUDIT = True` (this sets `ALLOW_TEST_AUDIT=1` for the run; each DS2 read is logged).
4. `PART = "generators"` first (about the full generator budget per seed and family), then `"classifiers"`
   with the banks from the first zip attached (`BANKS_DATASET_DIR`), or `"all"` in one long session.
5. Download the zip in Output and run `python kaggle/ingest_runs.py <zip>`; then
   `python -m ecgaug.experiments.phase4_leakage_audit --config configs/phase4_leakage_audit/audit.yaml --part report`.

In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
CONFIRM_AUDIT = False                 # True = allow this run to read DS2 (ALLOW_TEST_AUDIT=1); only after the final evaluation
PART = "generators"                   # "generators", "classifiers", "all" or "report"
BANKS_DATASET_DIR = ""                # for PART = "classifiers": the attached zip contents with data/synthetic/audit/
SMOKE_TEST = False                    # True = quick check on fake data (a few minutes, no results, no DS2)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"
PUSH_RESULTS = False                  # True = commit and push the results to GIT_REF from Kaggle; default: ingest the zip yourself
GIT_NAME = ""                         # your name for the results commit, e.g. "Akash Saha"
GIT_EMAIL = ""                        # your GitHub email for the results commit

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase4_leakage_audit/audit.yaml"]
extra = ["--part", PART, "--device", DEVICE]
if SMOKE_TEST:
    extra = ["--smoke"]
if CONFIRM_AUDIT and not SMOKE_TEST:
    env["ALLOW_TEST_AUDIT"] = "1"     # the audit refuses to read DS2 without it, and before the final evaluation
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase4_leakage_audit", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs]
cmd += ["--raw-dir", RAW_DATASET_DIR]   # the MIT-BIH files are copied from here and checked against the committed checksums
if BANKS_DATASET_DIR:
    cmd += ["--artifacts-dir", BANKS_DATASET_DIR]
if PUSH_RESULTS and not SMOKE_TEST:
    branch = GIT_REF if len(GIT_REF) != 40 else "kaggle-results"
    cmd += ["--push-to", branch, "--git-name", GIT_NAME, "--git-email", GIT_EMAIL]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO, env=env).returncode   # env carries the token for the push
print("finished with exit code", code, "(2 = the run was refused, read the message above)")